# AMS-SkipGNN Kaggle Stage 3

Upload **this** notebook: `notebooks/ams_skipgnn_kaggle_stage3.ipynb`.

Use **GPU T4**, Internet **ON**, then **Save Version → Save & Run All**.

Clones `aryonmt/finalProject` branch **`feat/three-new-architectures`**. Override with `REPO_BRANCH`.

Default **`STAGE=3`**. Do not set `STAGE=2` (that would retrain PPI/GDI SkipGNN/AMS).

| STAGE | What runs |
| --- | --- |
| `0` | Smoke: DTI `--quick` |
| `1` | DTI + DDI full (skip if CSV exists) |
| `2` | Cached Stage 1, then PPI + GDI + DTI ablation/robustness |
| `3` (default) | Fill missing fig5 bars, train `gat` / `3hop` / `contrastive` on DTI, dump embeddings for t-SNE |

**Stage 3 fills these missing comparison bars, then trains the new models:**

1. Heuristic **hard** AUPRC on DTI and DDI (old Stage 1 rows had uniform-only)
2. **GCN** on PPI and GDI, uniform **and** hard, 3 seeds
3. New models on DTI: `gat`, `3hop`, `contrastive` (3 seeds) + checkpoints/embeddings
4. Seed-42 embeddings for GCN / SkipGNN / AMS on DTI and GDI (paper-style t-SNE)

Cached Stage 1/2 CSVs in the clone are reused. New rows are **merged**, not overwritten.

Last cell writes **one** zip: `/kaggle/working/ams_skipgnn_kaggle_bundle.zip`


In [ ]:
import os, sys, platform, subprocess, shutil
from pathlib import Path

print('python', sys.version)
print('platform', platform.platform())
try:
    import torch
    print('torch', torch.__version__, 'cuda', torch.cuda.is_available())
    if torch.cuda.is_available():
        print('gpu', torch.cuda.get_device_name(0))
except Exception as e:
    print('torch import failed', e)

REPO = 'https://github.com/aryonmt/finalProject.git'
BRANCH = os.environ.get('REPO_BRANCH', 'feat/three-new-architectures')
WORK = Path('/kaggle/working') if Path('/kaggle/working').exists() else Path.cwd()
ROOT = WORK / 'finalProject'
if (Path.cwd() / 'src' / 'models').exists():
    ROOT = Path.cwd()
    print('already in repo', ROOT)
else:
    if ROOT.exists():
        shutil.rmtree(ROOT)
    subprocess.check_call(['git', 'clone', '--depth', '1', '--branch', BRANCH, REPO, str(ROOT)])
    print('cloned', ROOT, 'branch', BRANCH)
os.chdir(ROOT)
sys.path.insert(0, str(ROOT))
print('cwd', os.getcwd())


In [ ]:
import subprocess, sys
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-e', '.', '-q'])
print('pip install -e . done')
subprocess.check_call([sys.executable, 'scripts/fetch_data.py'])
print('data fetch done')


In [ ]:
import subprocess, sys
rc = subprocess.call([sys.executable, '-m', 'pytest', '-q', 'tests/test_smoke.py'])
print('pytest rc', rc)
assert rc == 0, 'smoke tests failed'


In [ ]:
import os, subprocess, sys, time
from pathlib import Path

stage = os.environ.get('STAGE', '3')
print('STAGE', stage, '(0=smoke, 1=DTI+DDI, 2=PPI/GDI extras, 3=gaps + new models + t-SNE)')
t0 = time.time()

if stage in {'0', '1', '2'}:
    datasets_to_check = ['DTI'] if stage == '0' else ['DTI', 'DDI']
    for ds in datasets_to_check:
        csv_path = Path(f'results/{ds}/benchmark.csv')
        if csv_path.exists() and stage != '0':
            print(f'[{ds}] Found existing benchmark.csv, skipping baseline retrain')
        else:
            cmd = [sys.executable, 'scripts/run_benchmark.py', '--dataset', ds, '--models', 'gcn', 'skipgnn', 'ams', 'heuristic', '--device', 'auto']
            if stage == '0':
                cmd += ['--quick']
            print('running', cmd)
            subprocess.check_call(cmd)
else:
    print('STAGE 3 skips DTI/DDI GCN/SkipGNN/AMS retrain; cached CSVs stay')

print('DTI/DDI gate done in', round((time.time()-t0)/60, 2), 'min')


In [ ]:
import os, subprocess, sys
stage = os.environ.get('STAGE', '3')
if stage != '2':
    print('skipping Stage 2 extras in STAGE=%s' % stage)
else:
    print('=== STAGE 2: PPI + GDI + ablation/robustness ===')
    subprocess.check_call([sys.executable, 'scripts/run_benchmark.py', '--dataset', 'PPI', '--models', 'skipgnn', 'ams', 'heuristic'])
    subprocess.check_call([sys.executable, 'scripts/run_benchmark.py', '--dataset', 'GDI', '--models', 'skipgnn', 'ams', 'heuristic'])
    subprocess.check_call([sys.executable, 'scripts/run_ablation.py', '--dataset', 'DTI'])
    subprocess.check_call([sys.executable, 'scripts/run_robustness.py', '--dataset', 'DTI'])
print('Stage 2 cell done')


In [ ]:
import os, subprocess, sys
from pathlib import Path
import pandas as pd

stage = os.environ.get('STAGE', '3')


def _csv(ds):
    return Path(f'results/{ds}/benchmark.csv')


def _has_model(ds, model, need_hard=False):
    path = _csv(ds)
    if not path.exists():
        return False
    df = pd.read_csv(path)
    sub = df[df['model'].astype(str).str.lower() == model.lower()]
    if sub.empty:
        return False
    if not need_hard:
        return True
    return 'hard_auprc' in sub.columns and bool(sub['hard_auprc'].notna().any())


def _run(cmd):
    print('running', cmd, flush=True)
    subprocess.check_call(cmd)


if stage != '3':
    print('skipping STAGE 3 work; set STAGE=3')
else:
    py = sys.executable
    print('=== STAGE 3a: missing fig5 bars ===')
    if not _has_model('DTI', 'heuristic', need_hard=True):
        _run([py, 'scripts/run_benchmark.py', '--dataset', 'DTI', '--models', 'heuristic'])
    else:
        print('[DTI] heuristic hard already present, skip')
    if not _has_model('DDI', 'heuristic', need_hard=True):
        _run([py, 'scripts/run_benchmark.py', '--dataset', 'DDI', '--models', 'heuristic'])
    else:
        print('[DDI] heuristic hard already present, skip')
    if not _has_model('PPI', 'gcn'):
        _run([py, 'scripts/run_benchmark.py', '--dataset', 'PPI', '--models', 'gcn'])
    else:
        print('[PPI] gcn already present, skip')
    if not _has_model('GDI', 'gcn'):
        _run([py, 'scripts/run_benchmark.py', '--dataset', 'GDI', '--models', 'gcn', '--save-embeddings'])
    else:
        print('[GDI] gcn already present, skip')

    print('=== STAGE 3b: new architectures on DTI ===')
    new_models = []
    for name in ('gat', '3hop', 'contrastive'):
        if not _has_model('DTI', name):
            new_models.append(name)
    if new_models:
        _run([py, 'scripts/run_benchmark.py', '--dataset', 'DTI', '--models', *new_models, '--save-embeddings', '--save-checkpoints'])
    else:
        print('[DTI] gat/3hop/contrastive already present, skip')

    print('=== STAGE 3c: seed-42 embeddings for t-SNE (GCN/SkipGNN/AMS) ===')

    def _has_emb(ds, model, seed=42):
        return Path(f'results/{ds}/embeddings_{model}_seed{seed}.npz').exists()

    dti_need = [m for m in ('gcn', 'skipgnn', 'ams') if not _has_emb('DTI', m)]
    if dti_need:
        _run([py, 'scripts/run_benchmark.py', '--dataset', 'DTI', '--models', *dti_need, '--seeds', '42', '--save-embeddings'])
    else:
        print('[DTI] seed-42 embeddings already present, skip')
    gdi_need = [m for m in ('gcn', 'skipgnn', 'ams') if not _has_emb('GDI', m)]
    if gdi_need:
        _run([py, 'scripts/run_benchmark.py', '--dataset', 'GDI', '--models', *gdi_need, '--seeds', '42', '--save-embeddings'])
    else:
        print('[GDI] seed-42 embeddings already present, skip')
    subprocess.call([py, 'scripts/plot_tsne.py'])

print('Stage 3 cell done')


In [ ]:
import json, os, shutil, subprocess, sys, zipfile
from datetime import datetime, timezone
from pathlib import Path

subprocess.call([sys.executable, 'scripts/plot_tsne.py'])
subprocess.call([sys.executable, 'scripts/make_figures.py'])

repo = Path.cwd()
work = Path('/kaggle/working') if Path('/kaggle/working').exists() else repo
staging = work / '_kaggle_bundle_staging'
if staging.exists():
    shutil.rmtree(staging)
staging.mkdir(parents=True)

def copy_tree(src: Path, dest: Path) -> int:
    if not src.exists():
        return 0
    n = 0
    dest.mkdir(parents=True, exist_ok=True)
    for path in src.rglob('*'):
        if path.is_file() and path.name not in {'.gitkeep', '.DS_Store'}:
            target = dest / path.relative_to(src)
            target.parent.mkdir(parents=True, exist_ok=True)
            shutil.copy2(path, target)
            n += 1
    return n

n_results = copy_tree(repo / 'results', staging / 'results')
n_figures = copy_tree(repo / 'figures', staging / 'figures')

nb_candidates = [
    Path('/kaggle/working/__notebook__.ipynb'),
    Path('/kaggle/working/__notebook_source__.ipynb'),
    work / 'ams_skipgnn_kaggle_stage3.ipynb',
    repo / 'notebooks' / 'ams_skipgnn_kaggle_stage3.ipynb',
    work / 'ams_skipgnn_kaggle_runner.ipynb',
    repo / 'notebooks' / 'ams_skipgnn_kaggle_runner.ipynb',
]
nb_src = next((p for p in nb_candidates if p.is_file()), None)
if nb_src is not None:
    dest_nb = staging / 'notebooks' / 'ams_skipgnn_kaggle_stage3.ipynb'
    dest_nb.parent.mkdir(parents=True, exist_ok=True)
    shutil.copy2(nb_src, dest_nb)

files = sorted(p.relative_to(staging).as_posix() for p in staging.rglob('*') if p.is_file())
manifest = {
    'created_utc': datetime.now(timezone.utc).strftime('%Y-%m-%dT%H:%M:%SZ'),
    'stage': os.environ.get('STAGE', '3'),
    'cwd': str(repo),
    'n_result_files': n_results,
    'n_figure_files': n_figures,
    'notebook_source': str(nb_src) if nb_src else None,
    'files': files,
    'import_map': {
        'results/': 'results/',
        'figures/': 'figures/',
        'notebooks/ams_skipgnn_kaggle_stage3.ipynb': 'notebooks/ams_skipgnn_kaggle_stage3.ipynb',
    },
}
try:
    import torch
    manifest['torch'] = torch.__version__
    manifest['cuda'] = bool(torch.cuda.is_available())
    if torch.cuda.is_available():
        manifest['gpu'] = torch.cuda.get_device_name(0)
except Exception:
    pass
(staging / 'MANIFEST.json').write_text(json.dumps(manifest, indent=2), encoding='utf-8')
(staging / 'IMPORT.txt').write_text(
    'Drop this zip at the repo root. Unpack results/, figures/, and notebooks/ over the repo.\n',
    encoding='utf-8',
)

zip_path = work / 'ams_skipgnn_kaggle_bundle.zip'
if zip_path.exists():
    zip_path.unlink()
with zipfile.ZipFile(zip_path, 'w', zipfile.ZIP_DEFLATED) as zf:
    for path in staging.rglob('*'):
        if path.is_file():
            zf.write(path, path.relative_to(staging).as_posix())
shutil.rmtree(staging, ignore_errors=True)

print('DOWNLOAD THIS FILE:', zip_path)
print('bytes', zip_path.stat().st_size)
print('files', files)
print('KAGGLE STAGE 3 COMPLETE — download only ams_skipgnn_kaggle_bundle.zip')
